In [1]:
import numpy as np
from scipy import special
import pandas as pd

### True Underlying Environment

In [2]:
def set_environment_contextual(
    n,
    arms,
    mu_a,
    tau_a,
    theta_ctx_mu,
    theta_ctx_tau,
    sigma_a,
    env_seed=0,
):
    """
    Generate participant-level true reward parameters for a 1-binary-context model.

    Indexing convention:
    - i indexes participants
    - a indexes arms (a=0 for arm 0, a=1 for arm 1, etc.)

    True mean reward for participant i and arm a:
        E[r_{i,a} | c_i] = theta0[i, a] + theta1[i, a] * c_i

    IMPORTANT: mu/tau vectors are arm-indexed, not context-indexed.
    """
    mu_a = np.asarray(mu_a, dtype=float)
    tau_a = np.asarray(tau_a, dtype=float)
    theta_ctx_mu = np.asarray(theta_ctx_mu, dtype=float)
    theta_ctx_tau = np.asarray(theta_ctx_tau, dtype=float)
    sigma_a = np.asarray(sigma_a, dtype=float)

    env_rng = np.random.default_rng(env_seed)

    # Baseline (intercept) term per participant and arm: theta0[i, a] ~ N(mu_a[a], tau_a[a])
    theta0 = env_rng.normal(loc=mu_a, scale=tau_a, size=(n, arms))

    # Context coefficient per participant and arm: theta1[i, a] ~ N(theta_ctx_mu[a], theta_ctx_tau[a])
    theta1 = env_rng.normal(loc=theta_ctx_mu, scale=theta_ctx_tau, size=(n, arms))

    return theta0, theta1, sigma_a


def sample_binary_contexts(n, p_one=0.2, context_seed=0):
    """Generate participant-specific binary contexts c_i in {0,1}."""
    rng = np.random.default_rng(context_seed)
    return rng.binomial(1, p_one, size=n).astype(float)

### Functions

In [3]:
def normal_cdf(z):
    z = np.asarray(z, dtype=float)
    return 0.5 * (1.0 + special.erf(z / np.sqrt(2.0)))

# w_i = 1 / (s_i^2 + tau^2)
# mu_hat = sum_i w_i c_i / sum_i w_i
# profile log-likelihood:
# ℓ(τ²) = -½ Σ_i [ log(s_i² + τ²) + (c_i - μ̂)² / (s_i² + τ²) ]
def profile_ll_and_mu(c, s2, tau2):
    w = 1.0 / (s2 + tau2)
    mu = np.sum(w * c) / np.sum(w)
    ll = -0.5 * np.sum(np.log(s2 + tau2) + (c - mu) ** 2 * w)
    return ll, mu

# PROFILE maximum likelihood estimator 
def fit_mu_tau2(c, s2, grid=None):
    c = np.asarray(c, float)
    s2 = np.asarray(s2, float)
    if grid is None:
        base = float(np.var(c)) + float(np.mean(s2))
        hi = max(1e-6, 10.0 * base)
        grid = np.logspace(-8, np.log10(hi), 120)

    best_tau2 = 0.0
    best_ll, best_mu = profile_ll_and_mu(c, s2, 0.0)
    for tau2 in grid:
        ll, mu = profile_ll_and_mu(c, s2, tau2)
        if ll > best_ll:
            best_ll, best_mu, best_tau2 = ll, mu, tau2
    return best_mu, best_tau2


def _reward_mean(theta0, theta1, a_idx, c_t):
    return theta0[np.arange(theta0.shape[0]), a_idx] + theta1[np.arange(theta1.shape[0]), a_idx] * c_t

### Learning Algorithms

Exact same as before.

In [4]:
def thompson_sampling(times, participants, n_arms, theta, sigma_r, rng, mu0, tau0):
    sigma_r = np.asarray(sigma_r, float)
    mu0 = np.asarray(mu0, float)
    tau0 = np.asarray(tau0, float)

    N = np.zeros((participants, n_arms), int)
    S = np.zeros((participants, n_arms), float)
    actions = np.empty((participants, times), int)
    rewards = np.empty((participants, times), float)

    sig2_arr = (
        float(sigma_r) ** 2
        if sigma_r.size == 1 else sigma_r.astype(float) ** 2
    )

    prec_prior = 1.0 / (tau0 ** 2)

    for t in range(times):
        precision_data = N / sig2_arr
        post_var = 1.0 / (precision_data + prec_prior)
        post_mean = post_var * (mu0 / (tau0 ** 2) + S / sig2_arr)

        mask = (N == 0)
        post_mean = np.where(mask, mu0[np.newaxis, :], post_mean)
        post_var = np.where(mask, (tau0 ** 2)[np.newaxis, :], post_var)

        sampled = rng.normal(loc=post_mean, scale=np.sqrt(post_var))
        a_star = np.argmax(sampled, axis=1)

        idx = np.arange(participants)
        locs = theta[idx, a_star]
        sigs = float(sigma_r) if sigma_r.size == 1 else sigma_r[a_star]
        r = rng.normal(loc=locs, scale=sigs)

        actions[:, t] = a_star
        rewards[:, t] = r
        N[idx, a_star] += 1
        S[idx, a_star] += r

    return N, S, actions, rewards


def pooled_thompson_sampling(times, participants, n_arms, theta, sigma_r, rng, mu0, tau0):
    sigma_r = np.asarray(sigma_r, float)
    mu0 = np.asarray(mu0, float)
    tau0 = np.asarray(tau0, float)

    N = np.zeros(n_arms, int)
    S = np.zeros(n_arms, float)
    actions = np.empty((participants, times), int)
    rewards = np.empty((participants, times), float)

    sig2_arr = (
        float(sigma_r) ** 2
        if sigma_r.size == 1 else sigma_r ** 2
    )
    prec_prior = 1.0 / (tau0 ** 2)

    for t in range(times):
        precision_data = N / sig2_arr
        post_var = 1.0 / (precision_data + prec_prior)
        post_mean = post_var * (mu0 / (tau0 ** 2) + S / sig2_arr)

        sampled = rng.normal(loc=post_mean, scale=np.sqrt(post_var))
        a_star = int(np.argmax(sampled))

        r = rng.normal(
            loc=theta[:, a_star],
            scale=(float(sigma_r) if sigma_r.size == 1 else sigma_r[a_star]),
            size=participants,
        )

        actions[:, t] = a_star
        rewards[:, t] = r
        N[a_star] += participants
        S[a_star] += r.sum()

    return N, S, actions, rewards


def learning_empirical_bayes(
    times,
    participants,
    n_arms,
    theta,
    sigma_arm,
    rng,
    mu0,
    tau0,
    grid=None,
):
    sigma_arm = np.asarray(sigma_arm, float)
    mu0 = np.asarray(mu0, float)
    tau0 = np.asarray(tau0, float)

    m = np.broadcast_to(mu0, (participants, n_arms)).copy()
    v = np.broadcast_to(tau0 ** 2, (participants, n_arms)).copy()

    N = np.zeros((participants, n_arms), int)
    actions = np.empty((participants, times), int)
    rewards = np.empty((participants, times), float)

    for t in range(times):
        mu_hat = np.empty(n_arms)
        tau2_hat = np.empty(n_arms)
        for a in range(n_arms):
            mu_hat[a], tau2_hat[a] = fit_mu_tau2(m[:, a], v[:, a], grid)

        lam = np.where(
            tau2_hat > 0,
            tau2_hat / (tau2_hat + v),
            1.0,
        )
        m_eb = lam * m + (1.0 - lam) * mu_hat
        v_eb = np.where(
            tau2_hat > 0,
            (v * tau2_hat) / (v + tau2_hat),
            0.0,
        )

        if n_arms == 2:
            diff = m_eb[:, 1] - m_eb[:, 0]
            var_diff = v_eb[:, 1] + v_eb[:, 0]
            p1 = normal_cdf(diff / np.sqrt(np.maximum(var_diff, 1e-20)))
            a_sel = (rng.random(participants) < p1).astype(int)
        else:
            sample = rng.normal(loc=m_eb, scale=np.sqrt(np.maximum(v_eb, 0.0)))
            a_sel = np.argmax(sample, axis=1)

        idx = np.arange(participants)
        locs = theta[idx, a_sel]
        sigs = sigma_arm[a_sel]
        r = rng.normal(loc=locs, scale=sigs)

        actions[:, t] = a_sel
        rewards[:, t] = r
        N[idx, a_sel] += 1

        sig2 = sigma_arm[a_sel] ** 2
        prec = 1.0 / v[idx, a_sel] + 1.0 / sig2
        v_new = 1.0 / prec
        m_new = v_new * (m[idx, a_sel] / v[idx, a_sel] + r / sig2)

        m[idx, a_sel] = m_new
        v[idx, a_sel] = v_new

    return m, v, N, actions, rewards

### Evaluation

In [5]:
def cumulative_regret(theta, actions):
    n, T = actions.shape
    best = theta.max(axis=1, keepdims=True)
    chosen = theta[np.arange(n)[:, None], actions]
    inst = best - chosen
    return np.cumsum(inst, axis=1)


def evaluate_contextual(theta0, theta1, sigma_a, contexts, T, mu0, tau0, seed=123):
    # Context enters only through the true reward means; learners stay context-unaware.
    contexts = np.asarray(contexts, float)
    theta = theta0 + theta1 * contexts[:, np.newaxis]
    n, arms = theta.shape

    rng_ts = np.random.default_rng(seed + 2)
    rng_pool_ts = np.random.default_rng(seed + 3)
    rng_eb = np.random.default_rng(seed + 4)

    N_ts, S_ts, actions_ts, rewards_ts = thompson_sampling(
        T, n, arms, theta, sigma_a, rng_ts, mu0=mu0, tau0=tau0
    )

    N_pts, S_pts, actions_pts, rewards_pts = pooled_thompson_sampling(
        T, n, arms, theta, sigma_a, rng_pool_ts, mu0=mu0, tau0=tau0
    )

    m_eb, v_eb, N_eb, actions_eb, rewards_eb = learning_empirical_bayes(
        T, n, arms, theta, sigma_a, rng_eb, mu0=mu0, tau0=tau0
    )

    return {
        "actions_thompson": actions_ts,
        "actions_pooled_thompson": actions_pts,
        "actions_eb": actions_eb,
        "regret_thompson": cumulative_regret(theta, actions_ts),
        "regret_pooled_thompson": cumulative_regret(theta, actions_pts),
        "regret_eb": cumulative_regret(theta, actions_eb),
        "rewards_thompson": rewards_ts,
        "rewards_pooled_thompson": rewards_pts,
        "rewards_eb": rewards_eb,
        "N_ts": N_ts,
        "S_ts": S_ts,
        "N_pooled_ts": N_pts,
        "S_pooled_ts": S_pts,
        "m_eb": m_eb,
        "v_eb": v_eb,
        "N_eb": N_eb,
    }

### Smoke Test

In [11]:
# Minimal smoke test: contextual environment, context-unaware learners (same model class as 022726).
n = 100
arms = 2
T = 200

# Parameter vectors below are indexed by ARM: [arm0, arm1], not by context.
# theta0 is the baseline term; theta1 is the context coefficient term.
theta0, theta1, sigma_a = set_environment_contextual(
    n=n,
    arms=arms,
    mu_a=[0.0, 0.0],          # mean of theta0 for [arm0, arm1]
    tau_a=[0.01, 0.01],         # std dev of theta0 for [arm0, arm1]
    theta_ctx_mu=[0, 0],  # mean of theta1 for [arm0, arm1]
    theta_ctx_tau=[0.01, 0.01], # std dev of theta1 for [arm0, arm1]
    sigma_a=[1, 1],       # reward noise std dev for [arm0, arm1]
    env_seed=888,
)

# contexts[i] = c_i in {0,1}; p_one is P(c_i = 1).
contexts = sample_binary_contexts(n, p_one=0.2, context_seed=1)

# Prior mean/std for context-unaware learners, by arm: [arm0, arm1].
mu0 = np.array([0.0, 0.0], dtype=float)
tau0 = np.array([0.5, 0.5], dtype=float)

out = evaluate_contextual(
    theta0=theta0,
    theta1=theta1,
    sigma_a=sigma_a,
    contexts=contexts,
    T=T,
    mu0=mu0,
    tau0=tau0,
    seed=888,
)

# For participant i and arm a, true mean reward is theta0[i,a] + theta1[i,a] * c_i.
# So when c_i=0: mean = theta0[i,a]; when c_i=1: mean = theta0[i,a] + theta1[i,a].

def final_group_regret(regret_matrix, mask):
    if np.sum(mask) == 0:
        return np.nan
    return regret_matrix[mask, -1].mean()

mask0 = contexts == 0
mask1 = contexts == 1

print("Final mean cumulative regret (overall):")
print("thompson:", out["regret_thompson"][:, -1].mean())
print("pooled_thompson:", out["regret_pooled_thompson"][:, -1].mean())
print("eb:", out["regret_eb"][:, -1].mean())

Final mean cumulative regret (overall):
thompson: 1.2576613183997047
pooled_thompson: 1.4079803439599323
eb: 1.4259471906275762
